### マルチモーダルAI APIサーバーの構成

L4 GPU（24GB VRAM）を活用して、以下の処理を行うAPIサーバーを構築します。

1. **フレームワーク**: `FastAPI` (高速な非同期APIサーバー)
2. **LLM推論 (Qwen 2.5 14B)**: `vLLM` (L4のVRAMに最適化して高速推論)
3. **メディア生成 (画像・音楽など)**: `diffusers` や `AudioCraft`
4. **音声分離・解析**: `Demucs` (ボーカル抽出), `librosa` / `basic-pitch` (MIDI/楽譜生成)

まずは必要なライブラリをインストールします。

In [17]:
import os

# 事前にColab上に配置・アップロードされたファイルのパスを指定します
# （例: 左側のファイルブラウザからアップロードした音声ファイルなど）
target_file_path = "/content/sample_audio.wav"

if os.path.exists(target_file_path):
    print(f"対象ファイルを確認しました: {target_file_path}")
    # ここにファイルを開く処理や、APIサーバーへの送信処理などを追加できます
else:
    print(f"ファイルが見つかりません: {target_file_path}")
    print("左側のファイルブラウザからファイルをアップロードするか、パスを修正してください。")


ファイルが見つかりません: /content/sample_audio.wav
左側のファイルブラウザからファイルをアップロードするか、パスを修正してください。


In [3]:
# サーバー構築に必要なライブラリ群のインストール（仮想的なセットアップ）
# ※ 実際のサーバー環境(Ubuntu等)で実行することを想定しています。
!pip install fastapi uvicorn pydantic python-multipart
!pip install vllm transformers diffusers
!pip install demucs librosa
!pip install basic-pitch --no-deps

  Using cached demucs-4.1.0-py3-none-any.whl.metadata (20 kB)
  Using cached julius-0.2.8-py3-none-any.whl.metadata (7.6 kB)
  Using cached lameenc-1.9.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_34_x86_64.whl.metadata (940 bytes)
  Using cached sphn-0.2.1-cp313-cp313-manylinux_2_24_x86_64.whl.metadata (275 bytes)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 146.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [demucs]
  Using cached basic_pitch-0.4.0-py2.py3-none-any.whl.metadata (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.3/758.3 kB 27.7 MB/s  0:00:00


### サーバーアプリケーションのスケルトンコード (FastAPI)

以下のコードは、要求された2つのフロー（プロンプト改良＋メディア生成、音声解析＋楽譜生成）を実装するFastAPIサーバーのひな形です。実際のL4サーバー上で `python server.py` のようにして起動します。

In [12]:
%%writefile server.py
import os
# Colab環境でのvLLM初期化エラーを防ぐためV1エンジンを無効化
os.environ["VLLM_USE_V1"] = "0"

from fastapi import FastAPI, UploadFile, File, HTTPException
from pydantic import BaseModel
import uvicorn
from vllm import LLM, SamplingParams
# import demucs.api
# from basic_pitch.inference import predict

app = FastAPI(title="Multi-modal AI Server", description="Powered by Qwen 2.5 (14B) AWQ")

# ==========================================
# 1. モデルの初期化
# ==========================================
print("Loading Qwen 2.5 (14B) AWQ model for API Server...")
llm = LLM(
    model="Qwen/Qwen2.5-14B-Instruct-AWQ",
    quantization="awq",
    tensor_parallel_size=1,
    gpu_memory_utilization=0.7,
    max_model_len=4096,
    trust_remote_code=True,
    enforce_eager=True
)

class PromptRequest(BaseModel):
    user_prompt: str
    media_type: str # "image", "video", "bgm", "se"

# ==========================================
# フロー1: プロンプト改良 -> メディア生成
# ==========================================
@app.post("/generate-media/")
async def generate_media(request: PromptRequest):
    # 1. Qwen 2.5 (14B) によるプロンプトの具体化と明確化
    system_prompt = f"{request.media_type}を生成するための詳細な英語プロンプトに改良してください。"
    prompt = f"<|im_start|>user\n{system_prompt}\n対象: {request.user_prompt}<|im_end|>\n<|im_start|>assistant\n"

    sampling_params = SamplingParams(temperature=0.7, max_tokens=256)
    outputs = llm.generate([prompt], sampling_params)
    refined_prompt = outputs[0].outputs[0].text.strip()

    # 2. 各種メディアの生成 (diffusers等へのルーティング)
    generated_media_url = ""
    if request.media_type == "image":
        # 画像生成処理 (例: Stable Diffusion 3 / SDXL)
        generated_media_url = "http://server/outputs/image.png"
    elif request.media_type == "bgm":
        # 音楽生成処理 (例: AudioLDM2 / MusicGen)
        generated_media_url = "http://server/outputs/bgm.wav"

    # 3. アプリケーションへ返す
    return {
        "original_prompt": request.user_prompt,
        "refined_prompt": refined_prompt,
        "media_url": generated_media_url
    }

# ==========================================
# フロー2: 音声入力 -> 解析・ボーカル抽出 -> 楽譜(MIDI)生成
# ==========================================
@app.post("/analyze-audio/")
async def analyze_audio(file: UploadFile = File(...)):
    # 1. 音声ファイルの保存と受け取り
    file_path = f"/tmp/{file.filename}"
    with open(file_path, "wb") as buffer:
        buffer.write(await file.read())

    # 2. 音声の解析・ボーカル抽出 (Demucsなどを使用)
    # separator = demucs.api.Separator()
    # origin, separated = separator.separate_audio_file(file_path)
    # vocal_audio = separated["vocals"]

    # 3. Qwen2.5 (Audio/VLMモデル) による解析やメタデータの抽出
    # （省略）

    # 4. ボーカル音声から楽譜(MIDI)ファイルの生成 (Basic Pitch等)
    # model_output, midi_data, note_events = predict(vocal_audio)
    midi_url = "http://server/outputs/score.mid"

    # 5. アプリケーションへ返す
    return {
        "status": "success",
        "vocal_extracted": True,
        "midi_url": midi_url,
        "analysis_summary": "ボーカル抽出および楽譜生成が完了しました。"
    }

if __name__ == "__main__":
    # L4インスタンス上で起動
    uvicorn.run(app, host="0.0.0.0", port=8000)


Overwriting server.py


### L4 GPUサーバーにおけるメモリ設計の注意点

NVIDIA L4はVRAMが**24GB**です。
* **Qwen 2.5 (14B)** をFP16（半精度）でロードすると約28GB必要となり、24GBに収まりません。
* **対策**: Qwen 2.5 (14B) を動かす場合は、**INT8 または INT4（AWQ, GPTQなど）の量子化モデル**を使用してください。INT4量子化であれば約8〜10GBのVRAMで動作するため、残りの14GBで画像生成(Stable Diffusion)や音声分離(Demucs)のモデルを同時にメモリに載せて動かすことが可能になります。

### AWQ量子化モデルの読み込み実装

vLLMを使ってQwen 2.5 (14B) のAWQモデルを読み込むための具体的なコードです。

重要なポイントは以下の3点です：
1. **`model`**: Hugging Faceの `Qwen/Qwen2.5-14B-Instruct-AWQ` を指定します。
2. **`quantization="awq"`**: AWQ形式であることを明示します。
3. **`gpu_memory_utilization`**: vLLMはデフォルトでGPUメモリの90%を確保してしまいます。他のメディア生成モデル（DiffusersやDemucsなど）を同じL4 GPUに載せるために、この値を `0.4` 〜 `0.5`（約10〜12GB）程度に制限してVRAMに空きを作ります。

In [10]:
%%writefile test_awq.py
import os
# Colab環境でのvLLM初期化エラーを防ぐためV1エンジンを無効化
os.environ["VLLM_USE_V1"] = "0"

from vllm import LLM, SamplingParams

def test_qwen_awq():
    print("Loading Qwen 2.5 (14B) AWQ model...")

    # AWQモデルの初期化
    llm = LLM(
        model="Qwen/Qwen2.5-14B-Instruct-AWQ",
        quantization="awq",
        tensor_parallel_size=1,
        gpu_memory_utilization=0.7,  # VRAMの70%(約16.8GB)を使用
        max_model_len=4096,
        trust_remote_code=True,
        enforce_eager=True  # メモリのオーバーヘッドとCUDAグラフ初期化エラーを回避
    )

    # サンプリングパラメータの設定
    sampling_params = SamplingParams(
        temperature=0.7,
        max_tokens=256
    )

    # テスト推論
    prompt = "<|im_start|>user\nマルチモーダルAIの魅力を一言で教えてください。<|im_end|>\n<|im_start|>assistant\n"
    print("\nRunning inference...")
    outputs = llm.generate([prompt], sampling_params)

    # 結果の表示
    for output in outputs:
        generated_text = output.outputs[0].text
        print(f"\n[Generated Text]:\n{generated_text}")

if __name__ == "__main__":
    test_qwen_awq()


Overwriting test_awq.py


先ほど作成した `server.py` のモック部分（モデルの初期化）を、上記の `LLM(...)` のコードに置き換えることで、APIサーバー上で実際にQwen 2.5の量子化モデルを稼働させることができます。

※ 実際の推論テストを行う場合は、L4 GPUインスタンス上で `!python test_awq.py` を実行してください。（初回はモデルのダウンロードに時間がかかります）

In [11]:
!python test_awq.py

Loading Qwen 2.5 (14B) AWQ model...
(MainProcess pid=13852) INFO 10-25 03:00:42 [api_utils.py:286] non-default args: {'trust_remote_code': True, 'max_model_len': 4096, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'model': 'Qwen/Qwen2.5-14B-Instruct-AWQ'}
(MainProcess pid=13852) WARNING 10-25 03:00:42 [interface.py:1457] Unknown vLLM environment variable detected: VLLM_USE_V1
(MainProcess pid=13852) INFO 10-25 03:00:43 [model.py:733] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=13852) INFO 10-25 03:00:43 [model.py:2152] Using max model len 4096
(MainProcess pid=13852) INFO 10-25 03:00:45 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=8192.
Parse safetensors files: 100% 3/3 [00:00<00:00,  3.59it/s]
(MainProcess pid=13852) WARNING 10-25 03:00:46 [vllm.py:1703] Enforce eager set, disabling torch.compile, CUDAGraphs, and JIT kernel warmup. This is equivalent to setting -cc.mode=none -cc.cudagra